In [4]:
import pandas as pd

rating_data = pd.read_csv('data/ratings_movies.csv')
rating_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Unnamed: 0  100836 non-null  int64  
 1   userId      100836 non-null  int64  
 2   movieId     100836 non-null  int64  
 3   rating      100836 non-null  float64
 4   date        100836 non-null  str    
 5   title       100836 non-null  str    
 6   genres      100836 non-null  str    
dtypes: float64(1), int64(3), str(3)
memory usage: 5.4 MB


In [5]:
import re 
def get_year_release(arg):
    #находим все слова по шаблону "(DDDD)"
    candidates = re.findall(r'\(\d{4}\)', arg) 
    # проверяем число вхождений
    if len(candidates) > 0:
        #если число вхождений больше 0,
	#очищаем строку от знаков "(" и ")"
        year = candidates[0].replace('(', '')
        year = year.replace(')', '')
        return int(year)
    else:
        #если год не указан, возвращаем None
        return None

In [6]:
rating_data['year_release'] = rating_data['title'].apply(
    get_year_release
)
#У скольких не указан год выпуска
len(rating_data[rating_data['year_release'].isna()])

18

In [7]:
rating_data['year_release']

0         1995.0
1         1995.0
2         1995.0
3         1995.0
4         1995.0
           ...  
100831    2017.0
100832    2017.0
100833    2017.0
100834    2017.0
100835    2017.0
Name: year_release, Length: 100836, dtype: float64

In [8]:
# id фильма из 1999 года с наимменьшим средним рейтингом
rating_data[rating_data['year_release'] == 1999]\
    .groupby('movieId')['rating'].mean().idxmin()

np.int64(145951)

In [9]:
rating_data[rating_data['movieId'] == 145951].head(1)['title']

100797    Bloodsport: The Dark Kumite (1999)
Name: title, dtype: str

In [10]:
'''
Какое сочетание жанров фильмов (genres), выпущенных в 2010 году,
получило наименьшую среднюю оценку (rating)?
'''
rating_data[rating_data['year_release'] == 2010].groupby('genres')['rating'].mean()\
.idxmin()

'Action|Sci-Fi'

In [11]:
'''
Какой пользователь (userId) посмотрел наибольшее
количество различных (уникальных) комбинаций жанров (genres) фильмов?
В качестве ответа запишите идентификатор этого пользователя.
'''
unique_genres_per_user = rating_data.groupby('userId')['genres'].nunique()
unique_genres_per_user.idxmax()

np.int64(599)

In [12]:
#пользователи,выставившие наименьшее количество оценок
count_over_avg = rating_data\
.groupby('userId')['rating'].agg(['count','mean'])


In [13]:
min_count = count_over_avg['count'].min()
count_over_avg[count_over_avg['count'] == min_count]['mean'].idxmax()

np.int64(53)

In [14]:
# можно вот так
count_over_avg.sort_values(['count', 'mean'], ascending=[True, False]).iloc[0]

count    20.0
mean      5.0
Name: 53, dtype: float64

In [15]:
'''
Найдите сочетание жанров (genres) за 2018 год,
которое имеет наибольший средний рейтинг (среднее по столбцу rating),
и при этом число выставленных ему оценок (количество значений в столбце rating) больше 10.
'''
stat = rating_data[rating_data['year_release'] == 2018]\
.groupby('genres')['rating']\
.agg(['mean', 'count'])

stat = stat[stat['count'] > 10]
stat = stat.sort_values(by='mean', ascending=False)
stat

,mean,count
genres,,
Action|Adventure|Sci-Fi,3.928571,14
Action|Comedy|Sci-Fi,3.875000,12


In [16]:
'''
Добавьте в таблицу новый признак year_rating — год выставления оценки. Создайте сводную таблицу, которая иллюстрирует зависимость среднего рейтинга фильма от года выставления оценки и жанра. Выберите верные варианты ответа, исходя из построенной таблицы:
'''
rating_data['year_rating'] = pd.to_datetime(rating_data['date']).dt.year
pivot = rating_data.pivot_table(
    index='year_rating',
    columns='genres',
    values='rating',
    aggfunc='mean'
)
pivot['Action|Adventure|Animation|Children|Comedy|IMAX'].idxmax()

np.int32(2013)

In [17]:
pivot.loc[2018][pivot.loc[2018] == pivot.loc[2018].max()]
# в топе есть  Animation|Children|Mystery

genres
Action|Comedy|Documentary                    5.0
Action|Crime|Sci-Fi|Thriller                 5.0
Action|Crime|Thriller|Western                5.0
Adventure|Children|Comedy|Fantasy|Mystery    5.0
Adventure|Fantasy|Musical                    5.0
Animation|Children|Mystery                   5.0
Animation|Comedy|Drama                       5.0
Animation|Comedy|Drama|Fantasy|Sci-Fi        5.0
Animation|Comedy|Fantasy|Musical             5.0
Animation|Crime|Drama                        5.0
Comedy|Crime|Fantasy                         5.0
Comedy|Crime|Horror|Thriller                 5.0
Comedy|Drama|Fantasy|Mystery|Romance         5.0
Comedy|Fantasy|Horror|Musical|Thriller       5.0
Comedy|Horror|Mystery                        5.0
Crime|Fantasy|Horror                         5.0
Crime|Mystery                                5.0
Drama|Horror|Mystery|Sci-Fi|Thriller         5.0
Drama|Romance|War                            5.0
Mystery|Romance|Sci-Fi|Thriller              5.0
Romance|Sci-F

In [18]:
pivot['Comedy']

year_rating
1996    3.228571
1997    3.409091
1998    3.000000
1999    3.606061
2000    3.141291
2001    3.318408
2002    3.198556
2003    3.120066
2004    3.356877
2005    2.963325
2006    3.347534
2007    2.928187
2008    3.426667
2009    3.273292
2010    3.179825
2011    3.232877
2012    3.478477
2013    3.489474
2014    3.271429
2015    3.106183
2016    3.373431
2017    2.852668
2018    3.121296
Name: Comedy, dtype: float64

In [19]:
orders = pd.read_csv('data/orders.csv', sep=';')
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Дата создания  18 non-null     str  
 1   Order ID       18 non-null     int64
 2   ID Покупателя  18 non-null     int64
 3   Статус         18 non-null     str  
 4   Оплачен        18 non-null     str  
 5   Отменен        18 non-null     str  
 6   Отгружен       18 non-null     str  
 7   ID товара      18 non-null     int64
 8   Количество     18 non-null     int64
dtypes: int64(4), str(5)
memory usage: 1.4 KB


In [20]:
products = pd.read_csv('data/products.csv', sep=';')
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Product_ID  14 non-null     int64
 1   Name        14 non-null     str  
 2   Price       14 non-null     int64
 3   CURRENCY    14 non-null     str  
dtypes: int64(2), str(2)
memory usage: 580.0 bytes


In [21]:
'''
объединю orders и products по ключам orders.'ID товара' и products.'Product_ID'
LEFT JOIN
'''
orders_products = orders.merge(
    products, left_on='ID товара', right_on='Product_ID', how='left'
    )
print('Заказ, о котором нет информации о товаре')
display(orders_products[orders_products['Product_ID'].isnull()])

print('Заказ, для которого отмена')
display(orders_products[orders_products['Отменен'] == 'Да'])

Заказ, о котором нет информации о товаре


,Дата создания,Order ID,ID Покупателя,Статус,Оплачен,Отменен,Отгружен,ID товара,Количество,Product_ID,Name,Price,CURRENCY
17,01.01.2001 00:00:00,0,1,"Оплачен, формируется к отправке",Да,Нет,Нет,666,1,NaN,NaN,NaN,NaN


Заказ, для которого отмена


,Дата создания,Order ID,ID Покупателя,Статус,Оплачен,Отменен,Отгружен,ID товара,Количество,Product_ID,Name,Price,CURRENCY
6,08.11.2019 08:36:22,5,5,Отменён,Нет,Да,Нет,124,1,124.0,Носки беговые Camino,999.0,RUR


In [26]:
paid_products = orders_products[orders_products['Оплачен'] == 'Да']
display(paid_products)

,Дата создания,Order ID,ID Покупателя,Статус,Оплачен,Отменен,Отгружен,ID товара,Количество,Product_ID,Name,Price,CURRENCY
8,08.11.2019 08:36:22,3,8,"Оплачен, формируется к отправке",Да,Нет,Нет,103,3,103.0,"Носки Подарочные, муж",199.0,RUR
9,08.11.2019 08:36:22,3,8,"Оплачен, формируется к отправке",Да,Нет,Нет,104,3,104.0,"Носки Подарочные, жен",249.0,RUR
10,08.11.2019 08:36:21,2,7,Выполнен,Да,Нет,Да,74,1,74.0,Платье Ночная Жизнь XXXL,8999.0,RUR
11,08.11.2019 08:36:21,2,7,Выполнен,Да,Нет,Да,53,1,53.0,Штаны Цветочная Поляна,4999.0,RUR
12,08.11.2019 08:36:21,2,7,Выполнен,Да,Нет,Да,137,1,137.0,Гольфы детские Снегурочка,99.0,RUR
13,08.11.2019 08:36:21,2,7,Выполнен,Да,Нет,Да,47,1,47.0,Шатны Полосатый рейс,2999.0,RUR
14,08.11.2019 08:36:20,1,5,"Оплачен, формируется к отправке",Да,Нет,Нет,71,1,71.0,Платье Ночная Жизнь,7999.0,RUR
15,08.11.2019 08:36:20,1,5,"Оплачен, формируется к отправке",Да,Нет,Нет,86,1,86.0,"Носки Простые, муж",45.0,RUR
16,08.11.2019 08:36:20,1,5,"Оплачен, формируется к отправке",Да,Нет,Нет,51,1,51.0,Платье Аленький цветочек,4999.0,RUR
17,01.01.2001 00:00:00,0,1,"Оплачен, формируется к отправке",Да,Нет,Нет,666,1,NaN,NaN,NaN,NaN


In [27]:
paid_products['expense'] = paid_products['Price'] * paid_products['Количество']
#display(paid_products[['expense', 'Price', 'Количество','CURRENCY']])
#display(paid_products['CURRENCY'].value_counts())


In [30]:
best_customer = paid_products.groupby('ID Покупателя')['expense'].sum()
best_customer.idxmax()

np.int64(7)